## Training Neural Networks and Testing Frontier LLMs

In [4]:
# libraries
import os
from dotenv import load_dotenv
from huggingface_hub import login
from pricer.evaluator import evaluate
from pricer.items import Item
from litellm import completion
import numpy as np
from tqdm.notebook import tqdm
import csv

from sklearn.feature_extraction.text import HashingVectorizer
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from torch.optim.lr_scheduler import CosineAnnealingLR

In [5]:
# setting up the environment
LITE_MODE = False

load_dotenv(override=True)

hf_token = os.environ.get('HF_TOKEN')
login(hf_token, add_to_git_credential=True)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [6]:
# loading the dataset
username = 'ed-donner'
dataset = f"{username}/items_lite" if LITE_MODE else f"{username}/items_full"

# retrieving the train, test and validation data
train, val, test = Item.from_hub(dataset)

print(f"Loaded {len(train):,} training items, {len(val):,} validation items, {len(test):,}")

Loaded 800,000 training items, 10,000 validation items, 10,000


### Artificial Neural Networks

In [7]:
# writing the test set to a csv file
with open('human_in.csv', 'w', encoding='utf-8') as file:
    writer = csv.writer(file)

    for t in test[:100]:
        writer.writerow([t.summary, 0])

In [8]:
# reading back in 
human_predictions = []

with open('human_out.csv', 'r', encoding='utf-8') as file:
    reader = csv.reader(file)

    for row in reader:
        human_predictions.append(float(row[1]))

In [9]:
def human_pricer(item):
    index = test.index(item)
    return human_predictions[index]

In [10]:
human = human_pricer(test[0])
actual = test[0].price

print(f"Human predicted {human} for an item that actually costs {actual}")

Human predicted 120.0 for an item that actually costs 219.0


In [11]:
evaluate(human_pricer, test, size=100)

  0%|          | 0/100 [00:00<?, ?it/s]

$99 $184 $12 $15 $18 $10 $119 $135 $6 $270 $643 $329 $15 $26 $24 $18 $29 $25 $25 $53 $35 $126 $25 $127 $273 $398 $55 $6 $101 $51 $30 $5 $35 $9 $10 $419 $25 $11 $186 $33 $161 $51 $23 $155 $150 $4 $31 $18 $115 $82 $25 $111 $410 $75 $67 $34 $8 $10 $122 $28 $116 $17 $19 $60 $599 $60 $160 $355 $75 $34 $17 $2 $70 $76 $41 $9 $226 $5 $5 $4 $0 $7 $5 $74 $7 $10 $68 $74 $5 $3 $17 $45 $5 $16 $0 $153 $2 $122 $150 $355 

### Vanilla Neural Network

In [12]:
# preparing the documents and prices

y = np.array([float(item.price) for item in train])
documents = [item.summary for item in train]

In [13]:
np.random.seed(42)
vectorizer = HashingVectorizer(n_features=5000, stop_words='english', binary=True)

x = vectorizer.fit_transform(documents)

In [14]:
# Defining the neural network

class NeuralNetwork(nn.Module):
    def __init__(self, input_size):
        super(NeuralNetwork, self).__init__()
        self.layer1 = nn.Linear(input_size, 128)
        self.layer2 = nn.Linear(128, 64)
        self.layer3 = nn.Linear(64, 64)
        self.layer4 = nn.Linear(64, 64)
        self.layer5 = nn.Linear(64, 64)
        self.layer6 = nn.Linear(64, 64)
        self.layer7 = nn.Linear(64, 64)
        self.layer8 = nn.Linear(64, 1)
        self.relu = nn.ReLU()


    def forward(self, x):
        output1 = self.relu(self.layer1(x))
        output2 = self.relu(self.layer2(output1))
        output3 = self.relu(self.layer3(output2))
        output4 = self.relu(self.layer4(output3))
        output5 = self.relu(self.layer5(output4))
        output6 = self.relu(self.layer6(output5))
        output7 = self.relu(self.layer7(output6))
        output8 = self.relu(self.layer8(output7))

        return output8


In [15]:
# converting the data into PyTorch tensors
x_train_tensor = torch.FloatTensor(x.toarray())
y_train_tensor = torch.FloatTensor(y).unsqueeze(1)

# splitting the data into train and validation set
x_train, x_val, y_train, y_val = train_test_split(x_train_tensor, y_train_tensor, test_size=0.01, random_state=42)

# creating the loader
train_dataset = TensorDataset(x_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

# initialize the model
input_size = x_train_tensor.shape[1]
model = NeuralNetwork(input_size)

In [16]:
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Number of trainable parameters: {trainable_params:,}")

Number of trainable parameters: 669,249


In [17]:
# defining the loss function and optimizer

loss_function = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)


# epochs - 2
EPOCHS = 2

for epoch in range(EPOCHS):
    model.train()

    for batch_x, batch_y in tqdm(train_loader):
        optimizer.zero_grad()

        outputs = model(batch_x)
        loss = loss_function(outputs, batch_y)
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        val_outputs = model(x_val)
        val_loss = loss_function(val_outputs, y_val)

    print(f"Epochs [{epoch+1}/{EPOCHS}], Train Loss: {loss.item():.3f}, Val Loss: {val_loss.item():.3f}")

  0%|          | 0/12375 [00:00<?, ?it/s]

Epochs [1/2], Train Loss: 5609.240, Val Loss: 12400.751


  0%|          | 0/12375 [00:00<?, ?it/s]

Epochs [2/2], Train Loss: 12938.434, Val Loss: 10895.245


In [18]:
def neural_network(item):
    model.eval()

    with torch.no_grad():
        vector = vectorizer.transform([item.summary])
        vector = torch.FloatTensor(vector.toarray())

        result = model(vector)[0].item()

    return max(0, result)

In [19]:
evaluate(neural_network, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$125 $93 $24 $101 $42 $62 $26 $6 $39 $84 $151 $85 $55 $102 $88 $9 $14 $20 $91 $28 $56 $76 $1 $8 $214 $258 $252 $47 $78 $43 $114 $106 $67 $1 $61 $340 $53 $38 $109 $78 $147 $33 $28 $11 $43 $62 $47 $16 $47 $2 $28 $51 $191 $19 $45 $93 $35 $149 $2 $53 $88 $14 $35 $16 $297 $69 $0 $339 $28 $83 $9 $25 $97 $118 $16 $59 $40 $22 $28 $63 $44 $84 $109 $36 $18 $220 $38 $32 $11 $200 $25 $6 $6 $5 $47 $91 $6 $24 $116 $212 $13 $28 $19 $21 $15 $36 $64 $292 $4 $64 $31 $124 $88 $37 $24 $72 $32 $87 $20 $81 $27 $127 $92 $8 $82 $33 $30 $58 $48 $22 $63 $29 $75 $22 $121 $23 $108 $1 $113 $1 $17 $107 $38 $85 $45 $67 $43 $271 $52 $3 $16 $26 $2 $2 $30 $110 $73 $1 $76 $14 $2 $9 $16 $22 $271 $23 $42 $32 $20 $58 $31 $13 $238 $59 $16 $60 $42 $30 $13 $103 $27 $14 $124 $62 $40 $41 $60 $161 $54 $16 $31 $45 $8 $80 $52 $6 $82 $35 $5 $15 

### Using Frontier LLMs

In [23]:
def messages_for(item):
    message = f"Estimate the price of this product. Respond with the price, no explanation\n\n{item.summary}"

    return [{"role": "user", "content": message}]

In [24]:
print(test[0].summary)

Title: Excess V2 Distortion/Modulation Pedal  
Category: Music Pedals  
Brand: Old Blood Noise  
Description: A versatile pedal offering distortion and three modulation modes—delay, chorus, and harmonized fifths—with full control over signal routing and expression.  
Details: Features include separate gain, tone, and volume controls; time, depth, and volume per modulation; order switching, soft‑touch bypass, and expression jack for dynamic control.


In [26]:
print(messages_for(test[0]))

[{'role': 'user', 'content': 'Estimate the price of this product. Respond with the price, no explanation\n\nTitle: Excess V2 Distortion/Modulation Pedal  \nCategory: Music Pedals  \nBrand: Old Blood Noise  \nDescription: A versatile pedal offering distortion and three modulation modes—delay, chorus, and harmonized fifths—with full control over signal routing and expression.  \nDetails: Features include separate gain, tone, and volume controls; time, depth, and volume per modulation; order switching, soft‑touch bypass, and expression jack for dynamic control.'}]


In [ ]:
# using GPT

def gpt_4_1_nano(item):
    response = completion(model='openai/gpt-4.1-nano', messages=messages_for(item))

    return response.choices[0].message.content

In [28]:
gpt_4_1_nano(test[0])

'$200'

In [29]:
test[0].price

219.0

In [30]:
evaluate(gpt_4_1_nano, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$31 $34 $30 $20 $20 $80 $24 $65 $10 $870 $113 $29 $20 $9 $9 $8 $41 $0 $40 $31 $59 $26 $65 $75 $182 $304 $705 $5 $151 $64 $30 $25 $10 $50 $35 $19 $60 $31 $34 $13 $175 $35 $20 $105 $70 $0 $22 $13 $65 $52 $24 $110 $225 $0 $147 $16 $8 $40 $52 $13 $86 $3 $31 $40 $571 $10 $90 $295 $25 $44 $17 $13 $30 $6 $5 $16 $126 $0 $9 $4 $30 $3 $15 $74 $7 $10 $32 $56 $0 $21 $3 $15 $5 $20 $2 $108 $6 $7 $70 $225 $50 $3 $7 $11 $101 $32 $10 $375 $4 $99 $0 $136 $69 $58 $54 $180 $0 $7 $64 $47 $24 $411 $50 $34 $0 $10 $5 $81 $29 $64 $79 $13 $35 $0 $85 $0 $85 $0 $68 $62 $16 $50 $70 $12 $144 $118 $15 $390 $15 $13 $3 $144 $12 $10 $3 $129 $101 $41 $30 $5 $411 $19 $7 $2 $440 $7 $752 $30 $15 $5 $10 $8 $70 $8 $22 $101 $2 $57 $36 $23 $546 $5 $150 $1 $5 $8 $63 $3 $10 $2 $25 $99 $25 $11 $40 $70 $30 $70 $21 $6 

In [35]:
def gpt_5_1(item):
    response = completion(model='openai/gpt-5.1', messages=messages_for(item), reasoning_effort='low', seed=42)

    return response.choices[0].message.content

In [36]:
evaluate(gpt_5_1, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$10 $74 $15 $20 $10 $150 $64 $115 $11 $50 $37 $21 $0 $14 $49 $3 $21 $20 $20 $39 $46 $14 $5 $25 $12 $223 $216 $5 $71 $64 $5 $30 $0 $55 $15 $119 $40 $38 $44 $16 $165 $40 $15 $65 $50 $5 $2 $1 $65 $48 $26 $110 $245 $0 $27 $44 $2 $49 $68 $1 $106 $48 $32 $65 $100 $1 $20 $305 $5 $54 $16 $3 $80 $6 $22 $18 $6 $2 $2 $5 $10 $4 $13 $74 $11 $25 $68 $56 $10 $13 $8 $25 $5 $10 $2 $88 $1 $97 $90 $255 $30 $3 $7 $19 $29 $72 $15 $345 $1 $139 $25 $116 $9 $53 $54 $20 $7 $5 $24 $296 $9 $41 $10 $26 $10 $20 $0 $1 $11 $59 $144 $7 $5 $5 $65 $2 $55 $10 $61 $12 $16 $449 $10 $7 $4 $38 $15 $110 $35 $8 $3 $84 $22 $50 $2 $51 $21 $41 $50 $5 $90 $19 $2 $2 $41 $2 $452 $25 $5 $2 $10 $2 $240 $13 $65 $41 $3 $37 $6 $2 $124 $15 $240 $59 $15 $3 $53 $17 $10 $2 $0 $1 $5 $111 $60 $50 $9 $30 $21 $6 